In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import sys
sys.path.append('../../')  # Add project root to path

import logging
from modules.base_ingestion import BaseBronzeIngestion
from typing import List, Dict, Any

# Log Configuration
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class DeputadosIngestion(BaseBronzeIngestion):
    """Concrete implementation for Deputados (Deputies) ingestion."""
    
    def __init__(self, spark, entity_name: str = 'deputados'):
        """Initialize calling base class with only Spark and entity name."""
        # Pass and spark and entity name to initialize base class
        super().__init__(
            spark=spark,
            entity_name=entity_name
        )
    
    def fetch_data(self) -> List[Dict[str, Any]]:
        """Implements the abstract method - uses _fetch_standard from base class."""
        logger.info(f"Starting {self.config.entity} ingestion pipeline...")
        
        # Prepare params: entity-specific + generic (idLegislatura)
        params = self.entity_config['params'].copy()
        idLegislaturas = self.generic_config['idLegislatura'].copy()
        
        # # Extract list of id legislaturas and iterate over each one
        # idLegislaturas = params_generic.pop('idLegislatura') if 'idLegislatura' in params else [None]
        all_data = []

        for id in idLegislaturas:
            if id:
                logger.info(f"Fetching {self.config.entity} for legislatura {id}...")
                params['idLegislatura'] = id
            else:
                logger.info(f"Fetching {self.config.entity} without year filter...")
            
            data = self._fetch_standard(params=params)
            
            if data:
                all_data.extend(data)
                logger.info(f"Retrieved {len(data)} records for legislatura {id}")
            else:
                logger.warning(f"No data found for legislatura {id}")
        
        logger.info(f"Total consolidated: {len(all_data)} records across {len(idLegislaturas)} legislatura(s)")
        return all_data

# Create ingestion instance
ingestion = DeputadosIngestion(spark=spark)

In [0]:
try:
    ingestion.execute(save_mode="overwrite")
    logger.info(f"{ingestion.config.entity} ingestion completed successfully!")
except Exception as e:
    logger.error(f"{ingestion.config.entity} ingestion failed: {str(e)}")
    raise

Verifyng the data ingestion values:

In [0]:
%sql
SELECT count(*) FROM workspace.camara_bronze.deputados

In [0]:
%sql
SELECT * FROM workspace.camara_bronze.deputados LIMIT 100;